# 09 - 手写 VLM（MiniVLM → Qwen2-VL → DeepSeek-OCR）

> 配套笔记：[01_03 视觉语言模型 VLM](https://github.com/Zoey-Cheng/MLSys-Learning-Notes/blob/main/notes/01_模型基础/01_03_VLM.md)

按 doc 顺序写三段可跑代码。**验证强度按模型分**：

| 模型 | 验证方式 | 强度 |
|---|---|---|
| MiniVLM (§2.1) | sentinel vision feature 自检 mask 替换正确性 | 正确性自检（LLaVA 是抽象不是 HF 单 class，无对齐目标） |
| **Qwen2-VL (§4.2)** | 在 §2.1 MiniVLM forward 骨架上加 **PatchMerger**（替代 MLP）+ **M-RoPE position_ids**；和 HF Qwen2-VL 对齐 | **逐元素数值对齐** ✓ |
| DeepSeek-OCR (§5.2) | shape + 16× 压缩比检查 | 仅做形状检查（DeepSeek-OCR 在 deepseek-ai 自家仓库，不在 HF 主仓） |

顺序：

1. MiniVLM (§2.1) — LLaVA 式 forward 骨架：mask 替换 vision feature 到占位位置
2. Qwen2-VL (§4.2) — PatchMerger（2×2 合并 + MLP, token ÷4）+ M-RoPE position_ids（(t, h, w) 三轴）
3. DeepSeek-OCR (§5.2) — DeepEncoder = SAM-ViT + 16× conv + CLIP-ViT 三段串行
4. 用 tiny HF Qwen2-VL 数值对齐：4a PatchMerger + 4b M-RoPE + 4c 整体 forward 骨架

## 0. 环境

`transformers>=4.45.0`（含 Qwen2-VL）。整体对齐用 tiny `Qwen2VLConfig` 实例化（参数 < 1M），不下载真权重。

In [21]:
!nvidia-smi

Mon Jun 29 05:56:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
!pip install -q "transformers>=4.45.0" torch

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. MiniVLM (§2.1) — LLaVA 式 forward

核心机制：按 `image_token_id` 用 mask 把视觉占位 embed 替换成 vision feature。`vision_tower` 和 `llm` 都 call 现成的（这里用 dummy），自己写的只有 MLP projector + ~30 行 forward 骨架。

末尾自检：构造 sentinel vision feature（全 7.0），forward 后检查 image 位置的 `inputs_embeds` 真的等于 projected sentinel——这就是 mask 替换是否正确的判据。

In [24]:
IMAGE_TOKEN_ID = 32000   # 视觉占位 token id (LLaVA 约定)

class DummyVisionTower(nn.Module):
    """模拟 CLIP/SigLIP image tower，输出 [B, N_patch, d_vision]"""
    def __init__(self, d_vision=1024, patch_size=14):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, d_vision, kernel_size=patch_size, stride=patch_size)
    def forward(self, pixel_values):                  # [B, 3, H, W]
        x = self.patch_embed(pixel_values)            # [B, d, H/p, W/p]
        return x.flatten(2).transpose(1, 2)           # [B, N, d_vision]

class DummyLLM(nn.Module):
    """模拟 LLaMA/Qwen 风格 LLM：embed + 一层 transformer + lm_head"""
    def __init__(self, vocab_size=32001, d_llm=512):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, d_llm)
        self.layer = nn.TransformerEncoderLayer(d_llm, nhead=8, dim_feedforward=2*d_llm, batch_first=True)
        self.lm_head = nn.Linear(d_llm, vocab_size)
    def forward(self, input_ids=None, inputs_embeds=None, **kwargs):
        x = inputs_embeds if inputs_embeds is not None else self.embed_tokens(input_ids)
        x = self.layer(x)
        return self.lm_head(x)

class MiniVLM(nn.Module):
    """= doc §2.1。LLaVA 式 forward：mask 替换占位 embed"""
    def __init__(self, vision_tower, llm, d_vision, d_llm):
        super().__init__()
        self.vision_tower = vision_tower                                          # 冻
        self.projector = nn.Sequential(                                           # 唯一从头训
            nn.Linear(d_vision, d_llm), nn.GELU(), nn.Linear(d_llm, d_llm))
        self.llm = llm                                                            # 冻 (初期)
    def encode_and_merge(self, input_ids, pixel_values):
        """分离出 embed-replacement 步骤以便单独验证"""
        img_feats = self.vision_tower(pixel_values)                               # [B, N_patch, d_vision]
        img_embeds = self.projector(img_feats)                                    # [B, N_patch, d_llm]
        inputs_embeds = self.llm.embed_tokens(input_ids)                          # [B, T, d_llm]
        mask = (input_ids == IMAGE_TOKEN_ID)
        inputs_embeds[mask] = img_embeds.reshape(-1, img_embeds.size(-1))
        return inputs_embeds, img_embeds, mask
    def forward(self, input_ids, pixel_values):
        inputs_embeds, _, _ = self.encode_and_merge(input_ids, pixel_values)
        return self.llm(inputs_embeds=inputs_embeds)

# 运行验证
d_vision, d_llm, patch_size = 1024, 512, 14
H = W = patch_size * 4                                # 4×4 patch grid
N_patch = (H // patch_size) ** 2                      # = 16
B = 2

vt = DummyVisionTower(d_vision, patch_size)
llm_dummy = DummyLLM(vocab_size=32001, d_llm=d_llm)
vlm = MiniVLM(vt, llm_dummy, d_vision, d_llm).eval()

pixel_values = torch.randn(B, 3, H, W)
prefix = torch.tensor([[1, 100, 200]] * B)
image_placeholders = torch.full((B, N_patch), IMAGE_TOKEN_ID)
suffix = torch.tensor([[300, 400, 500]] * B)
input_ids = torch.cat([prefix, image_placeholders, suffix], dim=1)
print(f'input_ids shape: {tuple(input_ids.shape)}  (含 {N_patch} 个 image 占位)')

with torch.no_grad():
    logits = vlm(input_ids, pixel_values)
print(f'logits shape   : {tuple(logits.shape)}  → [B, T, vocab]')

# 验证: mask 替换后, image 位置的 inputs_embeds 应等于 projected vision feature
with torch.no_grad():
    inputs_embeds, img_embeds, mask = vlm.encode_and_merge(input_ids, pixel_values)
img_positions  = inputs_embeds[mask]
expected_embed = img_embeds.reshape(-1, d_llm)
assert torch.allclose(img_positions, expected_embed)
print('\n[验证] mask 替换正确：image 位置的 inputs_embeds = projected vision feature')

input_ids shape: (2, 22)  (含 16 个 image 占位)
logits shape   : (2, 22, 32001)  → [B, T, vocab]

[验证] mask 替换正确：image 位置的 inputs_embeds = projected vision feature


## 2. Qwen2-VL (§4.2) — PatchMerger + M-RoPE

相对 §2.1 MiniVLM 的两处改动（forward 骨架不动）：

- **PatchMerger** 替代 §2.1 的两层 MLP——2×2 spatial 合并 + MLP，token 数 ÷4
- **M-RoPE position_ids**——文本侧三轴同号、图像侧 (t, h, w) 三轴坐标

两者都在下面 §4 用 HF Qwen2-VL 做逐元素对齐。

### 2.1 PatchMerger

= HF `Qwen2VLPatchMerger`。输入约定: 每 4 个连续 token = 一个 2×2 spatial block（真实 Qwen2-VL 在 vision_tower 内已排好）。

为能和 HF 数值对齐：**LayerNorm 用 `eps=1e-6`**（HF 默认值，和 PyTorch 默认 `1e-5` 不同）。

In [25]:
class PatchMerger(nn.Module):
    """= doc §3.3 / §4.2 / HF Qwen2VLPatchMerger
    输入约定: 每 4 个连续 token = 一个 2×2 spatial block (HF 在 vision_tower 内已排好)"""
    def __init__(self, d_vision, d_llm, spatial_merge=2):
        super().__init__()
        self.hidden = d_vision * spatial_merge ** 2                # = 4 * d_vision
        self.ln = nn.LayerNorm(d_vision, eps=1e-6)                 # ← 关键: HF 用 1e-6
        self.mlp = nn.Sequential(
            nn.Linear(self.hidden, self.hidden),
            nn.GELU(),
            nn.Linear(self.hidden, d_llm),
        )
    def forward(self, x):                                          # [N, d_vision]
        x = self.ln(x).view(-1, self.hidden)                       # [N/4, 4*d_vision]
        return self.mlp(x)                                         # [N/4, d_llm]

# Smoke test
merger_smoke = PatchMerger(1024, 4096).eval()
x = torch.randn(576, 1024)
with torch.no_grad():
    y = merger_smoke(x)
print(f'in : {tuple(x.shape)}  → out: {tuple(y.shape)}  (N ÷4)')
assert y.shape == (144, 4096)

in : (576, 1024)  → out: (144, 4096)  (N ÷4)


### 2.2 M-RoPE position_ids — (t, h, w) 三轴

= HF `Qwen2VLForConditionalGeneration.get_rope_index`。文本侧三轴同号、图像侧 (t, h, w) 三轴坐标 + 文本偏移。

**和 HF 对齐的两个关键约定**：

- `image_grid_thw` 存 **pre-merge** 网格（`(t, h_patch, w_patch)`）；函数内部按 `spatial_merge_size` 除 `h, w` 得 post-merge 网格
- 用 "running max + 1" 做 position 计数（不是简单累加 1）

简化：不识别 `vision_start/end_token_id`（HF 用它们标定 image 区段；这里直接靠 `image_token_id` 占位定位）；不处理 attention_mask 和 video。

In [26]:
def build_mrope_positions(input_ids, image_grid_thw, image_token_id, spatial_merge_size=2):
    """= doc §4.2 / 简化版 HF Qwen2VLModel.get_rope_index

    Args:
        input_ids: [B, T]
        image_grid_thw: tensor [num_images, 3], pre-merge (t, h, w)
        image_token_id: int
        spatial_merge_size: int, h/w 除以这个值得 post-merge 网格

    Returns: position_ids [3, B, T]"""
    B, T = input_ids.shape
    position_ids = torch.zeros(3, B, T, dtype=input_ids.dtype)
    img_idx = 0
    for b in range(B):
        running = 0                                    # = HF 的 "max + 1" 计数
        i = 0
        while i < T:
            if input_ids[b, i].item() != image_token_id:
                position_ids[:, b, i] = running        # 文本: 三轴同号
                i += 1
                running += 1
            else:
                t = image_grid_thw[img_idx, 0].item()
                h = image_grid_thw[img_idx, 1].item()
                w = image_grid_thw[img_idx, 2].item()
                # ↓ 关键: 对齐 HF 约定, 除 spatial_merge_size 得 post-merge 网格
                llm_t, llm_h, llm_w = t, h // spatial_merge_size, w // spatial_merge_size
                num = llm_t * llm_h * llm_w
                # 三轴 grid 坐标
                t_index = torch.arange(llm_t).repeat_interleave(llm_h * llm_w)
                h_index = torch.arange(llm_h).repeat_interleave(llm_w).repeat(llm_t)
                w_index = torch.arange(llm_w).repeat(llm_t * llm_h)
                position_ids[0, b, i:i+num] = running + t_index
                position_ids[1, b, i:i+num] = running + h_index
                position_ids[2, b, i:i+num] = running + w_index
                running += max(llm_t, llm_h, llm_w)    # 跳到 image 段后
                i += num
                img_idx += 1
    return position_ids

# 单独 smoke test (HF 对齐放 §5)
input_ids = torch.tensor([[10, 20, 30, 99, 99, 99, 99, 40, 50]])  # 99 = image token
image_grid_thw = torch.tensor([[1, 4, 4]])    # pre-merge → post-merge 2×2 = 4 tokens
pos = build_mrope_positions(input_ids, image_grid_thw, image_token_id=99, spatial_merge_size=2)
print(f'position_ids [3, B, T]: {tuple(pos.shape)}')
print(f'  text 前 3 token (t/h/w): {pos[:, 0, :3].tolist()}')
print(f'  image 4 个 (t,h,w)     : {pos[:, 0, 3:7].tolist()}')
print(f'  text 后 2 token         : {pos[:, 0, 7:].tolist()}')

position_ids [3, B, T]: (3, 1, 9)
  text 前 3 token (t/h/w): [[0, 1, 2], [0, 1, 2], [0, 1, 2]]
  image 4 个 (t,h,w)     : [[3, 3, 3, 3], [3, 3, 4, 4], [3, 4, 3, 4]]
  text 后 2 token         : [[5, 6], [5, 6], [5, 6]]


## 3. DeepSeek-OCR (§5.2) — DeepEncoder

三段串行：**SAM-ViT (局部 window attn) → 16× conv 下采样 → CLIP-ViT (全局 attn)**。

DeepSeek-OCR forward 骨架完全沿用 §2.1 MiniVLM，**唯一替换 `self.vision_tower = DeepEncoder()`**——所以这里只写 DeepEncoder。

简化：SAM 段用普通 `TransformerEncoder` 占位（真实是 window attention，4096 tokens 规模下才算得动）；16× conv 真实就是 4×4 stride 4。无 HF 现成对齐目标（DeepSeek-OCR 在 deepseek-ai 自家仓库），仅做 shape + 压缩比检查。

In [27]:
class DeepEncoder(nn.Module):
    """= doc §5.2。SAM-ViT → 16× conv 下采样 → CLIP-ViT 三段串行"""
    def __init__(self, d_local=128, d_global=256,
                 n_local_layers=2, n_global_layers=2,
                 patch_size=16, downsample=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, d_local, kernel_size=patch_size, stride=patch_size)
        self.sam_blocks = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_local, 4, 4*d_local, batch_first=True),
            num_layers=n_local_layers)
        # 16× conv 下采样 = 4×4 stride 4 conv, 序列长度 ÷ 16 (核心压缩点)
        self.compress = nn.Conv2d(d_local, d_global, kernel_size=downsample, stride=downsample)
        self.clip_blocks = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_global, 8, 4*d_global, batch_first=True),
            num_layers=n_global_layers)
    def forward(self, pixel_values):                              # [B, 3, H, W]
        x = self.patch_embed(pixel_values)                        # [B, d_local, H/16, W/16]
        B, d, h, w = x.shape
        x = x.flatten(2).transpose(1, 2)                          # [B, h*w, d_local]
        x = self.sam_blocks(x)
        x = x.transpose(1, 2).reshape(B, d, h, w)
        x = self.compress(x)                                      # [B, d_global, h/4, w/4]
        x = x.flatten(2).transpose(1, 2)
        x = self.clip_blocks(x)
        return x

# Run sanity (Base 1024² 配置等比缩小)
H = W = 256
B = 2
enc = DeepEncoder().eval()
pv = torch.randn(B, 3, H, W)
with torch.no_grad():
    feats = enc(pv)
N_in  = (H // 16) ** 2
N_out = N_in // 16
print(f'input          : {tuple(pv.shape)}')
print(f'patch tokens   : {N_in}  → SAM 段')
print(f'after 16× conv : {N_out}  → CLIP 段')
print(f'output         : {tuple(feats.shape)}  → 压缩比 {N_in // N_out}×')
assert feats.shape == (B, N_out, 256)
assert N_in // N_out == 16
print('\n[验证] output shape 与 16× 压缩比符合预期')

input          : (2, 3, 256, 256)
patch tokens   : 256  → SAM 段
after 16× conv : 16  → CLIP 段
output         : (2, 16, 256)  → 压缩比 16×

[验证] output shape 与 16× 压缩比符合预期


## 4. 用 tiny HF Qwen2-VL 数值对齐

仿 06_mini_moe `## 5a/5b/5c` 的三层对齐：

- **4a. PatchMerger** ↔ HF `Qwen2VLPatchMerger`：拷权重 + `allclose`
- **4b. M-RoPE** ↔ HF `get_rope_index`：同 input 逐元素对齐
- **4c. 整体 forward 骨架** ↔ HF `Qwen2VLForConditionalGeneration.forward`：用 tiny config 实例化真模型，手写骨架借 HF 的 `visual` / `model` / `lm_head` 作模块拼起来，logits `allclose`

> §1 MiniVLM 整体 forward / §3 DeepEncoder 没 HF 对齐目标（LLaVA 是抽象不是 HF 单 class、DeepSeek-OCR 在 deepseek-ai 自家仓库），各自用 mask-替换自检 + shape 自检兜底。

In [28]:
from transformers import Qwen2VLConfig, Qwen2VLForConditionalGeneration
from transformers.models.qwen2_vl.modeling_qwen2_vl import PatchMerger as HFPatchMerger

# Tiny config — 参数 < 1M, 不下载真权重
config = Qwen2VLConfig(
    vocab_size=200,
    hidden_size=64,
    intermediate_size=128,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    image_token_id=151,
    video_token_id=152,
    vision_start_token_id=150,
    vision_end_token_id=153,
    rope_scaling={"type": "mrope", "mrope_section": [4, 2, 2]},   # 三段和 = head_dim/2 = 16/2 = 8
    vision_config=dict(
        depth=2,
        embed_dim=32,
        hidden_size=64,
        num_heads=4,
        in_channels=3,
        patch_size=14,
        spatial_merge_size=2,
        temporal_patch_size=2,
    ),
)
ref = Qwen2VLForConditionalGeneration(config).eval()
sms = config.vision_config.spatial_merge_size
print(f'tiny Qwen2-VL params: {sum(p.numel() for p in ref.parameters())/1e6:.2f}M')

tiny Qwen2-VL params: 0.19M


In [29]:
# === 4a. PatchMerger weight transfer + allclose ===
context_dim, dim_out = 1024, 4096
hf_merger = HFPatchMerger(dim=dim_out, context_dim=context_dim, spatial_merge_size=2).eval()
my_merger = PatchMerger(d_vision=context_dim, d_llm=dim_out, spatial_merge=2).eval()

# 拷权重 (层名映射 ln_q ↔ ln, mlp[0]/mlp[2] 同名)
my_merger.ln.load_state_dict(hf_merger.ln_q.state_dict())
my_merger.mlp[0].load_state_dict(hf_merger.mlp[0].state_dict())
my_merger.mlp[2].load_state_dict(hf_merger.mlp[2].state_dict())

x = torch.randn(16, context_dim)
with torch.no_grad():
    y_hf, y_mine = hf_merger(x), my_merger(x)
print(f'4a. PatchMerger diff: {(y_hf - y_mine).abs().max().item():.2e}')
assert torch.allclose(y_hf, y_mine, atol=1e-6)
print('    ✓ 数值一致')

4a. PatchMerger diff: 0.00e+00
    ✓ 数值一致


In [30]:
import torch

# === 4b. M-RoPE 函数对齐 ===
# input: text [10] + vision_start [150] + 4 个 image patch [151] + vision_end [153] + text [40]
# 总长度 = 8
t, h, w = 1, 4, 4
n_post = t * (h // sms) * (w // sms)
input_ids = torch.tensor([[10, 150] + [151] * n_post + [153, 40]])
image_grid_thw = torch.tensor([[t, h, w]])

# Construct mm_token_type_ids: 关键！只有 image_token_id (151) 标记为 1
# start (150) 和 end (153) 在 HF 逻辑里属于 type 0 (text)
mm_token_type_ids = torch.zeros_like(input_ids, dtype=torch.long)
mm_token_type_ids[input_ids == ref.config.image_token_id] = 1

# Create attention_mask
attention_mask = torch.ones_like(input_ids, dtype=torch.long)

# HF: get_rope_index
get_rope = getattr(ref, 'get_rope_index', None) or ref.model.get_rope_index
ref_pos, _ = get_rope(input_ids, mm_token_type_ids=mm_token_type_ids, image_grid_thw=image_grid_thw, attention_mask=attention_mask)

# Mine
my_pos = build_mrope_positions(input_ids, image_grid_thw,
                                image_token_id=ref.config.image_token_id, spatial_merge_size=sms)

print(f'4b. position_ids shape: ref {tuple(ref_pos.shape)}, mine {tuple(my_pos.shape)}')
print(f'    HF  pos[:, 0, :]: \n{ref_pos[:, 0, :].tolist()}')
print(f'    Mine pos[:, 0, :]: \n{my_pos[:, 0, :].tolist()}')
assert torch.equal(ref_pos, my_pos), 'M-RoPE position_ids 不一致'
print('    ✓ 逐元素一致')

4b. position_ids shape: ref (3, 1, 8), mine (3, 1, 8)
    HF  pos[:, 0, :]: 
[[0, 1, 2, 2, 2, 2, 4, 5], [0, 1, 2, 2, 3, 3, 4, 5], [0, 1, 2, 3, 2, 3, 4, 5]]
    Mine pos[:, 0, :]: 
[[0, 1, 2, 2, 2, 2, 4, 5], [0, 1, 2, 2, 3, 3, 4, 5], [0, 1, 2, 3, 2, 3, 4, 5]]
    ✓ 逐元素一致


In [31]:
# === 4c. 整体 forward 骨架对齐 ===
tps = config.vision_config.temporal_patch_size
ps  = config.vision_config.patch_size
n_pre = t * h * w
pixel_values = torch.randn(n_pre, 3 * tps * ps * ps)

# Construct mm_token_type_ids for HF
mm_token_type_ids_ref = torch.zeros_like(input_ids, dtype=torch.long)
mm_token_type_ids_ref[input_ids == ref.config.image_token_id] = 1
attention_mask_ref = torch.ones_like(input_ids, dtype=torch.long)

# Reference: HF 整体 forward
with torch.no_grad():
    ref_out = ref(input_ids=input_ids, pixel_values=pixel_values, image_grid_thw=image_grid_thw, mm_token_type_ids=mm_token_type_ids_ref, attention_mask=attention_mask_ref)
ref_logits = ref_out.logits

# Mine: 手写 forward 骨架
def my_qwen2vl_forward(ref, input_ids, pixel_values, image_grid_thw):
    # 1. 提取视觉特征
    visual_tower = getattr(ref, 'visual', None) or ref.model.visual
    image_embeds = visual_tower(pixel_values, grid_thw=image_grid_thw).last_hidden_state

    # 2. 关键：调用 PatchMerger (2x2 合并 + 投影) 使 token 数 ÷4，维度 32→64
    # HF 的 PatchMerger 通常在 visual.merger 中
    image_embeds = visual_tower.merger(image_embeds)

    # 3. Embedding 查表与替换
    embed_tokens = ref.get_input_embeddings()
    inputs_embeds = embed_tokens(input_ids)
    mask = (input_ids == ref.config.image_token_id)
    inputs_embeds[mask] = image_embeds

    # 4. 生成 M-RoPE position_ids
    position_ids = build_mrope_positions(
        input_ids, image_grid_thw,
        image_token_id=ref.config.image_token_id,
        spatial_merge_size=ref.config.vision_config.spatial_merge_size,
    )

    mm_token_type_ids_my = torch.zeros_like(input_ids, dtype=torch.long)
    mm_token_type_ids_my[input_ids == ref.config.image_token_id] = 1
    attention_mask_my = torch.ones_like(input_ids, dtype=torch.long)

    # 5. LLM Forward
    outputs = ref.model(
        inputs_embeds=inputs_embeds,
        position_ids=position_ids,
        mm_token_type_ids=mm_token_type_ids_my,
        attention_mask=attention_mask_my
    )
    return ref.lm_head(outputs.last_hidden_state)

with torch.no_grad():
    my_logits = my_qwen2vl_forward(ref, input_ids, pixel_values, image_grid_thw)

print(f'4c. logits shape: ref {tuple(ref_logits.shape)}, mine {tuple(my_logits.shape)}')
print(f'    max abs diff: {(ref_logits - my_logits).abs().max().item():.2e}')
assert torch.allclose(my_logits, ref_logits, atol=1e-5)
print('    ✓ 整体 forward 骨架与 HF 数值一致')

4c. logits shape: ref (1, 8, 200), mine (1, 8, 200)
    max abs diff: 0.00e+00
    ✓ 整体 forward 骨架与 HF 数值一致


## 注：与真实模型 / 笔记的差异

**对齐强度总览**（按模型）

| 模型 | 验证 | 强度 |
|---|---|---|
| MiniVLM (§2.1) | sentinel vision feature → 检查 image 位置 embed | 正确性自检 |
| **Qwen2-VL (§4.2)** | tiny HF 真模型 → 4a 拷权重 + `allclose` / 4b `torch.equal` / 4c 整体 logits `allclose` | **逐元素数值对齐** ✓ |
| DeepSeek-OCR (§5.2) | shape + 16× 压缩比 | 形状检查（不在 HF 主仓） |

**主要简化**

- §1 MiniVLM 的 `DummyVisionTower` / `DummyLLM` 替代真 CLIP/SigLIP + Vicuna/Qwen，只验证形状和计算路径；和 §4 用 tiny HF Qwen2-VL 的整体对齐是两条独立路径
- M-RoPE 简化：不识别 `vision_start/end_token_id`（HF 用它们框定 image 区段；这里靠 `image_token_id` 占位）、不处理 attention_mask、不支持 video。在简单单图 case 下输出和 HF 完全一致（4b 验证）
- DeepEncoder 的 SAM 段用普通 `TransformerEncoder` 占位（真实 SAM 用 window attention 在 4096 tokens 规模下才算得动）；CLIP 段同理；DeepSeek-OCR 完整实现在 deepseek-ai 仓库不在 HF transformers

**关键 detail**

- PatchMerger `LayerNorm` eps 必须显式写 `1e-6`（PyTorch 默认 1e-5，对齐会失败）
- HF `Qwen2VLConfig.rope_scaling.mrope_section` 三段和必须等于 `head_dim/2`（这里 head_dim=16, 取 `[4, 2, 2]` 和 = 8 = 16/2；真实 Qwen2-VL-7B 是 `[16, 24, 24]` 和 = 64 = 128/2）
- tiny config 没 attention_mask，HF Qwen2 model 接受 `attention_mask=None` 即 full attention
- pixel_values 是 NaViT-flattened 格式 `[total_patches, 3 * tps * ps * ps]`，不是 `[B, 3, H, W]` ——真实 inference 用 `AutoProcessor` 预处理就行